# ETL‑script

### Importeer modules, maak databaseverbinding en definieer hulpfuncties

In [128]:
# Standard library
import sqlite3
from typing import List, Dict
import pandas as pd
from functools import lru_cache
from pathlib import Path

In [129]:
def connect_db(db_path: Path) -> sqlite3.Connection:
    # Verbind met de SQLite database
    try:
        con = sqlite3.connect(db_path)
        con.execute("PRAGMA foreign_keys = ON;")
        return con
    except sqlite3.Error as e:
        # Later logger
        print(f"Fout bij het verbinden met de database: {e}")
        raise

In [130]:
def prepare_target_schema(db_path: Path) -> None:
    # Maak of actualiseer de doelschema's in de database.
    # Dim_Product en Dim_Klant krijgen SCD2-kolommen voor historie-opbouw.

    con = connect_db(db_path)

    try:
        # Met 'with con:' wordt de transactie automatisch gecommit als alles goed gaat,
        # of gerollbacked (teruggedraaid) als er een fout optreedt.
        with con:
            con.executescript(
                """
                CREATE TABLE IF NOT EXISTS Dim_Product (
                    ProductKey     INTEGER PRIMARY KEY AUTOINCREMENT,
                    ProductNr      INTEGER NOT NULL,
                    ProductType    TEXT NOT NULL CHECK (ProductType IN ('Fiets','Accessoire')),
                    Naam           TEXT,
                    Merk           TEXT,
                    Soort          TEXT,
                    Type           TEXT,
                    Kleur          TEXT,
                    Standaardprijs REAL,
                    Inkoopprijs    REAL,
                    ValidFrom      TEXT NOT NULL,
                    ValidTo        TEXT,
                    IsCurrent      INTEGER NOT NULL DEFAULT 1 CHECK (IsCurrent IN (0,1))
                );

                CREATE TABLE IF NOT EXISTS Dim_Partner (
                    PartnerKey     INTEGER PRIMARY KEY AUTOINCREMENT,
                    PartnerNr      INTEGER NOT NULL,
                    PartnerType    TEXT NOT NULL CHECK (PartnerType IN ('Fabrikant','Leverancier')),
                    Naam           TEXT,
                    Adres          TEXT,
                    Plaats         TEXT,
                    UNIQUE (PartnerType, PartnerNr)
                );

                CREATE TABLE IF NOT EXISTS Dim_Klant (
                    KlantKey            INTEGER PRIMARY KEY AUTOINCREMENT,
                    KlantNr             INTEGER NOT NULL,
                    Naam                TEXT NOT NULL,
                    Adres               TEXT,
                    Woonplaats          TEXT,
                    Geslacht            TEXT CHECK (Geslacht IN ('M','V','X') OR Geslacht IS NULL),
                    Geboortedatum       TEXT,
                    Leeftijd            INTEGER,
                    Leeftijdscategorie  TEXT,
                    ValidFrom           TEXT NOT NULL,
                    ValidTo             TEXT,
                    IsCurrent           INTEGER NOT NULL DEFAULT 1 CHECK (IsCurrent IN (0,1))
                );

                CREATE TABLE IF NOT EXISTS Dim_Monteur (
                    MonteurKey     INTEGER PRIMARY KEY AUTOINCREMENT,
                    MonteurNr      INTEGER NOT NULL UNIQUE,
                    Naam           TEXT NOT NULL,
                    Woonplaats     TEXT,
                    Uurloon        REAL
                );

                CREATE TABLE IF NOT EXISTS Dim_Filiaal (
                    FiliaalKey     INTEGER PRIMARY KEY AUTOINCREMENT,
                    FiliaalNr      INTEGER NOT NULL UNIQUE,
                    Naam           TEXT NOT NULL,
                    Adres          TEXT,
                    Provincie      TEXT
                );

                CREATE TABLE IF NOT EXISTS Dim_Datum (
                    DatumKey       INTEGER PRIMARY KEY AUTOINCREMENT,
                    Datum          TEXT NOT NULL UNIQUE,
                    Dag            INTEGER NOT NULL,
                    Maand          INTEGER NOT NULL,
                    Jaar           INTEGER NOT NULL,
                    Kwartaal       INTEGER NOT NULL,
                    Weekdag        TEXT NOT NULL,
                    Seizoen        TEXT,
                    IsWeekend      INTEGER NOT NULL CHECK (IsWeekend IN (0,1))
                );

                CREATE TABLE IF NOT EXISTS Dim_Tijd (
                    TijdKey        INTEGER PRIMARY KEY AUTOINCREMENT,
                    Tijd           TEXT NOT NULL UNIQUE,
                    Uur            INTEGER NOT NULL,
                    Minuut         INTEGER NOT NULL,
                    Dagdeel        TEXT
                );

                CREATE TABLE IF NOT EXISTS Fact_Inkoop (
                    InkoopKey      INTEGER PRIMARY KEY AUTOINCREMENT,
                    InkoopNr       INTEGER NOT NULL UNIQUE,
                    ProductKey     INTEGER NOT NULL,
                    PartnerKey     INTEGER NOT NULL,
                    DatumKey       INTEGER NOT NULL,
                    Aantal         INTEGER NOT NULL,
                    Inkoopprijs    REAL NOT NULL,
                    Inkoopbedrag   REAL NOT NULL,
                    KortingBedrag  REAL,
                    FOREIGN KEY (ProductKey) REFERENCES Dim_Product(ProductKey),
                    FOREIGN KEY (PartnerKey) REFERENCES Dim_Partner(PartnerKey),
                    FOREIGN KEY (DatumKey)   REFERENCES Dim_Datum(DatumKey)
                );

                CREATE TABLE IF NOT EXISTS Fact_Verkoop (
                    VerkoopKey     INTEGER PRIMARY KEY AUTOINCREMENT,
                    VerkoopNr      INTEGER NOT NULL UNIQUE,
                    ProductKey     INTEGER NOT NULL,
                    KlantKey       INTEGER NOT NULL,
                    MonteurKey     INTEGER NOT NULL,
                    FiliaalKey     INTEGER NOT NULL,
                    DatumKey       INTEGER NOT NULL,
                    Aantal         INTEGER NOT NULL,
                    Verkoopprijs   REAL NOT NULL,
                    Omzet          REAL NOT NULL,
                    Inkoopbedrag   REAL,
                    Brutowinst     REAL,
                    FOREIGN KEY (ProductKey) REFERENCES Dim_Product(ProductKey),
                    FOREIGN KEY (KlantKey)   REFERENCES Dim_Klant(KlantKey),
                    FOREIGN KEY (MonteurKey) REFERENCES Dim_Monteur(MonteurKey),
                    FOREIGN KEY (FiliaalKey) REFERENCES Dim_Filiaal(FiliaalKey),
                    FOREIGN KEY (DatumKey)   REFERENCES Dim_Datum(DatumKey)
                );

                CREATE TABLE IF NOT EXISTS Fact_Onderhoud (
                    OnderhoudKey       INTEGER PRIMARY KEY AUTOINCREMENT,
                    OnderhoudNr        INTEGER NOT NULL UNIQUE,
                    ProductKey         INTEGER NOT NULL,
                    MonteurKey         INTEGER NOT NULL,
                    FiliaalKey         INTEGER NOT NULL,
                    DatumKey           INTEGER NOT NULL,
                    StartTijdKey       INTEGER NOT NULL,
                    EindTijdKey        INTEGER NOT NULL,
                    AantalOnderhoud    INTEGER NOT NULL DEFAULT 1,
                    OnderhoudsduurMin  INTEGER NOT NULL,
                    Arbeidskosten      REAL NOT NULL,
                    FOREIGN KEY (ProductKey)   REFERENCES Dim_Product(ProductKey),
                    FOREIGN KEY (MonteurKey)   REFERENCES Dim_Monteur(MonteurKey),
                    FOREIGN KEY (FiliaalKey)   REFERENCES Dim_Filiaal(FiliaalKey),
                    FOREIGN KEY (DatumKey)     REFERENCES Dim_Datum(DatumKey),
                    FOREIGN KEY (StartTijdKey) REFERENCES Dim_Tijd(TijdKey),
                    FOREIGN KEY (EindTijdKey)  REFERENCES Dim_Tijd(TijdKey)
                );
                """
            )

            # Maak partiele unieke indexen
            con.executescript(
                """
                CREATE UNIQUE INDEX IF NOT EXISTS UX_Dim_Product_Current
                ON Dim_Product (ProductType, ProductNr)
                WHERE IsCurrent = 1;

                CREATE UNIQUE INDEX IF NOT EXISTS UX_Dim_Klant_Current
                ON Dim_Klant (KlantNr)
                WHERE IsCurrent = 1;
                """
            )
    finally:
        con.close()

In [131]:
import datetime as dt

def normalize_time(time_str: str | None) -> str | None:
    # Verwijdert microseconden van een tijd-string
    if not time_str:
        return None
    return str(time_str).split(".")[0]

def minutes_between(start: str | None, end: str | None) -> int | None:
    # Berekent het aantal minuten tussen twee tijdstippen
    if not start or not end:
        return None

    try:
        fmt = "%H:%M:%S"
        start_dt = dt.datetime.strptime(normalize_time(start), fmt)
        end_dt = dt.datetime.strptime(normalize_time(end), fmt)

        # Voorkomt negatieve minuten als een taak over middernacht heen gaat
        if end_dt < start_dt:
            end_dt += dt.timedelta(days=1)

        return int((end_dt - start_dt).total_seconds() // 60)
    except ValueError:
        # Vangt fouten op als de tijd een ongeldig formaat heeft
        return None

def age_from_birthdate(date_str: str | None, ref_date: dt.date | None = None) -> int | None:
    #Berekent de leeftijd op basis van een geboortedatum (YYYY-MM-DD)
    if not date_str:
        return None

    if ref_date is None:
        ref_date = dt.date.today()

    try:
        # Haal de eerste 10 tekens (YYYY-MM-DD) op, negeer eventuele tijdstippen
        birth = dt.date.fromisoformat(str(date_str)[:10])
        age = ref_date.year - birth.year - ((ref_date.month, ref_date.day) < (birth.month, birth.day))
        return age
    except ValueError:
        return None

def age_category(age: int | None) -> str | None:
    # Deelt een leeftijd in een specifieke categorie in
    if age is None:
        return None
    if age < 18:
        return "<18"
    if age <= 30:
        return "18-30"
    if age <= 50:
        return "31-50"
    return "51+"

def season(month: int) -> str | None:
    # Bepaalt het seizoen op basis van het maandnummer (1-12)
    if month in (12, 1, 2):
        return "Winter"
    elif month in (3, 4, 5):
        return "Lente"
    elif month in (6, 7, 8):
        return "Zomer"
    elif month in (9, 10, 11):
        return "Herfst"
    return None

def dagdeel(hour: int) -> str | None:
    # Bepaalt het dagdeel op basis van het uur (0-23)
    if hour < 0 or hour > 23:
        return None
    if hour < 6:
        return "Nacht"  # Toegevoegd voor de volledigheid!
    elif hour < 12:
        return "Ochtend"
    elif hour < 18:
        return "Middag"
    else:
        return "Avond"

### Gegevens extraheren en transformeren vanuit het SDM

In [132]:
def extract_sdm_data(sdm_path: Path) -> dict[str, pd.DataFrame]:
    # Lees de bron-database (SDM) en bouw per doeltabel een DataFrame
    # dat past bij het Data Warehouse (DWH) schema.
    con = connect_db(sdm_path)

    try:
        # Als we 'Fietsverkoop_Monteur' een tweede keer opvragen, haalt Python het
        # direct uit het geheugen in plaats van de database opnieuw te bevragen!
        @lru_cache(maxsize=None)
        def read(table: str) -> pd.DataFrame:
            return pd.read_sql_query(f"SELECT * FROM {table}", con)

        # Dim_Product: combineer fiets‑ en accessoiregegevens uit alle bronnen
        def make_product_df() -> pd.DataFrame:
            cols = ["ProductType","ProductNr","Naam","Merk","Soort","Type","Kleur","Standaardprijs","Inkoopprijs"]
            bike_sources = [
                read("Fiets_Inkoop_Fiets").assign(ProductType="Fiets"),
                read("Fietsverkoop_Fiets").assign(ProductType="Fiets"),
                read("Onderhoud_Fiets").assign(ProductType="Fiets"),
            ]
            bike = pd.concat(bike_sources, ignore_index=True).drop_duplicates(subset=["ProductType","fietsnr"])
            bike = bike.rename(columns={
                "fietsnr":"ProductNr","naam":"Naam","merk":"Merk","soort":"Soort",
                "type":"Type","kleur":"Kleur","standaardprijs":"Standaardprijs",
                "inkoopprijs":"Inkoopprijs"
            })
            acc_sources = [
                read("Accessoire_Inkoop_Accessoire").assign(ProductType="Accessoire"),
                read("Accessoireverkoop_Accessoire").assign(ProductType="Accessoire"),
            ]
            acc = pd.concat(acc_sources, ignore_index=True).drop_duplicates(subset=["ProductType","accessoirenr"])
            acc = acc.rename(columns={
                "accessoirenr":"ProductNr","naam":"Naam","soort":"Soort",
                "standaardprijs":"Standaardprijs","inkoopprijs":"Inkoopprijs"
            })

            for df in [bike, acc]:
                for col in cols:
                    if col not in df.columns:
                        df[col] = None

            return pd.concat([bike[cols], acc[cols]], ignore_index=True).drop_duplicates()

        # Dim_Partner: fabrikanten en leveranciers combineren
        def make_partner_df() -> pd.DataFrame:
            cols = ["PartnerType","PartnerNr","Naam","Adres","Plaats"]
            fabrikanten = read("Fiets_Inkoop_Fabrikant").assign(PartnerType="Fabrikant")
            fabrikanten = fabrikanten.rename(columns={
                "fabrikantnr":"PartnerNr","naam":"Naam","adres":"Adres","plaats":"Plaats"
            })
            leveranciers = read("Accessoire_Inkoop_Leverancier").assign(PartnerType="Leverancier")
            leveranciers = leveranciers.rename(columns={
                "leveranciernr":"PartnerNr","naam":"Naam","adres":"Adres","woonplaats":"Plaats"
            })
            return pd.concat([fabrikanten[cols], leveranciers[cols]], ignore_index=True).drop_duplicates()

        # Dim_Klant: klanten samenvoegen en leeftijd berekenen
        def make_klant_df() -> pd.DataFrame:
            cols = ["KlantNr","Naam","Adres","Woonplaats","Geslacht","Geboortedatum","Leeftijd","Leeftijdscategorie"]
            kv = read("Fietsverkoop_Klant").rename(columns={
                "klantnr":"KlantNr","naam":"Naam","adres":"Adres",
                "woonplaats":"Woonplaats","geslacht":"Geslacht","geboortedatum":"Geboortedatum"
            })
            ka = read("Accessoireverkoop_Klant").rename(columns={
                "klantnr":"KlantNr","naam":"Naam","adres":"Adres",
                "woonplaats":"Woonplaats","geslacht":"Geslacht","geboortedatum":"Geboortedatum"
            })
            klanten = pd.concat([kv, ka], ignore_index=True)

            # Pandas geeft NaN (Not a Number) voor lege waarden. Onze try...except in age_from_birthdate
            # vangt dit netjes op, maar het is veiliger om expliciet te controleren op pd.isna()
            klanten["Leeftijd"] = klanten["Geboortedatum"].apply(
                lambda d: None if pd.isna(d) else age_from_birthdate(str(d))
            )
            klanten["Leeftijdscategorie"] = klanten["Leeftijd"].apply(age_category)
            klanten = klanten.sort_values(by=["KlantNr"]).drop_duplicates(subset=["KlantNr"], keep="last")
            return klanten[cols]

        # Dim_Monteur: unieke monteurs uit alle bronnen
        def make_monteur_df() -> pd.DataFrame:
            cols = ["MonteurNr","Naam","Woonplaats","Uurloon"]
            bronnen = [read("Fietsverkoop_Monteur"), read("Accessoireverkoop_Monteur"), read("Onderhoud_Monteur")]
            monteurs = pd.concat(bronnen, ignore_index=True).drop_duplicates(subset=["monteurnr"])
            monteurs = monteurs.rename(columns={
                "monteurnr":"MonteurNr","naam":"Naam","woonplaats":"Woonplaats","uurloon":"Uurloon"
            })
            return monteurs[cols]

        # Dim_Filiaal: filialen uit verkoop en onderhoud
        def make_filiaal_df() -> pd.DataFrame:
            cols = ["FiliaalNr","Naam","Adres","Provincie"]
            bronnen = [read("Fietsverkoop_Filiaal"), read("Accessoireverkoop_Filiaal"), read("Onderhoud_Filiaal")]
            filialen = pd.concat(bronnen, ignore_index=True).drop_duplicates(subset=["filiaalnr"])
            filialen = filialen.rename(columns={
                "filiaalnr":"FiliaalNr","naam":"Naam","adres":"Adres","provincie":"Provincie"
            })
            return filialen[cols]

        # Dim_Datum: verzamel alle datums uit verkoop, onderhoud en inkoop
        def make_datum_df() -> pd.DataFrame:
            bike_dates = read("Fietsverkoop_Fiets_Verkoop")["datum"].dropna().astype(str)
            acc_dates  = read("Accessoireverkoop_Accessoire_Verkoop")["datum"].dropna().astype(str)
            maintenance = read("Onderhoud")["datum"].dropna().astype(str)

            # Gebruik een veilige functie voor de inkoopdatums om ValueError te voorkomen bij lege data
            def build_date(r):
                try:
                    return f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01"
                except (ValueError, TypeError):
                    return None

            fiets_inkoop = read("Fiets_Inkoop").assign(Datum=lambda df: df.apply(build_date, axis=1))["Datum"]
            acc_inkoop  = read("Accessoire_Inkoop").assign(Datum=lambda df: df.apply(build_date, axis=1))["Datum"]

            all_dates = pd.concat([bike_dates, acc_dates, maintenance, fiets_inkoop, acc_inkoop], ignore_index=True)
            all_dates = pd.to_datetime(all_dates.dropna()).drop_duplicates().sort_values()

            df = pd.DataFrame({"Datum": all_dates.dt.strftime("%Y-%m-%d")})
            dt_values = pd.to_datetime(df["Datum"])
            df["Dag"] = dt_values.dt.day
            df["Maand"] = dt_values.dt.month
            df["Jaar"] = dt_values.dt.year
            df["Kwartaal"] = dt_values.dt.quarter
            df["Weekdag"] = dt_values.dt.day_name()
            df["Seizoen"] = df["Maand"].apply(lambda m: season(int(m)))
            df["IsWeekend"] = dt_values.dt.dayofweek.isin([5, 6]).astype(int)
            return df

        # Dim_Tijd: unieke start‑ en eindtijden omzetten naar hh:mm:ss
        def make_tijd_df() -> pd.DataFrame:
            tijden = pd.concat([
                read("Onderhoud")["starttijd"].apply(normalize_time),
                read("Onderhoud")["eindtijd"].apply(normalize_time)
            ], ignore_index=True).dropna().drop_duplicates().sort_values()

            df = pd.DataFrame({"Tijd": tijden})
            tijd_values = pd.to_datetime(df["Tijd"], format="%H:%M:%S")
            df["Uur"] = tijd_values.dt.hour
            df["Minuut"] = tijd_values.dt.minute
            df["Dagdeel"] = df["Uur"].apply(lambda h: dagdeel(int(h)))
            return df

        # Fact_Inkoop: fiets‑ en accessoire‑inkopen combineren en uniek nummer geven
        def make_fact_inkoop_df() -> pd.DataFrame:
            fi = read("Fiets_Inkoop").merge(read("Fiets_Inkoop_Fiets"), left_on="fiets", right_on="fietsnr")
            bike = pd.DataFrame({
                "InkoopNr": fi["inkoopnr"],
                "ProductType": "Fiets",
                "ProductNr": fi["fiets"],
                "PartnerType": "Fabrikant",
                "PartnerNr": fi["fabrikant"],
                "Datum": fi.apply(lambda r: f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01", axis=1),
                "Aantal": fi["aantal"].fillna(0),
                "Inkoopprijs": fi["inkoopprijs"].fillna(0),
                "Inkoopbedrag": (fi["aantal"].fillna(0) * fi["inkoopprijs"].fillna(0)).round(2),
                "KortingBedrag": ((fi["standaardprijs"].fillna(0) - fi["inkoopprijs"].fillna(0)) * fi["aantal"].fillna(0)).round(2),
            })

            ai = read("Accessoire_Inkoop").merge(read("Accessoire_Inkoop_Accessoire"), left_on="accessoire", right_on="accessoirenr")
            acc = pd.DataFrame({
                "InkoopNr": ai["inkoopnr"] + 1000000,
                "ProductType": "Accessoire",
                "ProductNr": ai["accessoire"],
                "PartnerType": "Leverancier",
                "PartnerNr": ai["leverancier"],
                "Datum": ai.apply(lambda r: f"{int(r['inkoopjaar']):04d}-{int(r['inkoopmaand']):02d}-01", axis=1),
                "Aantal": ai["aantal"].fillna(0),
                "Inkoopprijs": ai["inkoopprijs"].fillna(0),
                "Inkoopbedrag": (ai["aantal"].fillna(0) * ai["inkoopprijs"].fillna(0)).round(2),
                "KortingBedrag": ((ai["standaardprijs"].fillna(0) - ai["inkoopprijs"].fillna(0)) * ai["aantal"].fillna(0)).round(2),
            })
            return pd.concat([bike, acc], ignore_index=True)

        # Fact_Verkoop: fiets‑ en accessoireverkopen combineren en uniek nummer geven
        def make_fact_verkoop_df() -> pd.DataFrame:
            bikev = read("Fietsverkoop_Fiets_Verkoop").merge(
                read("Fietsverkoop_Fiets"), left_on="fiets", right_on="fietsnr"
            ).merge(
                read("Fietsverkoop_Monteur")[["monteurnr","filiaal"]], left_on="monteur", right_on="monteurnr"
            )
            bike_fact = pd.DataFrame({
                "VerkoopNr": bikev["fiets_verkoopnr"],
                "ProductType": "Fiets",
                "ProductNr": bikev["fiets"],
                "KlantNr": bikev["klant"],
                "MonteurNr": bikev["monteur"],
                "FiliaalNr": bikev["filiaal"],
                "Datum": bikev["datum"].astype(str),
                "Aantal": bikev["aantal"].fillna(0),
                "Verkoopprijs": bikev["verkoopprijs"].fillna(0),
                "Omzet": (bikev["aantal"].fillna(0) * bikev["verkoopprijs"].fillna(0)).round(2),
                "Inkoopbedrag": (bikev["aantal"].fillna(0) * bikev["inkoopprijs"].fillna(0)).round(2),
                "Brutowinst": ((bikev["aantal"].fillna(0) * bikev["verkoopprijs"].fillna(0)) - (bikev["aantal"].fillna(0) * bikev["inkoopprijs"].fillna(0))).round(2),
            })

            accv = read("Accessoireverkoop_Accessoire_Verkoop").merge(
                read("Accessoireverkoop_Accessoire"), left_on="accessoire", right_on="accessoirenr"
            ).merge(
                read("Accessoireverkoop_Monteur")[["monteurnr","filiaal"]], left_on="monteur", right_on="monteurnr"
            )
            acc_fact = pd.DataFrame({
                "VerkoopNr": accv["accessoire_verkoopnr"] + 1000000,
                "ProductType": "Accessoire",
                "ProductNr": accv["accessoire"],
                "KlantNr": accv["klant"],
                "MonteurNr": accv["monteur"],
                "FiliaalNr": accv["filiaal"],
                "Datum": accv["datum"].astype(str),
                "Aantal": accv["aantal"].fillna(0),
                "Verkoopprijs": accv["verkoopprijs"].fillna(0),
                "Omzet": (accv["aantal"].fillna(0) * accv["verkoopprijs"].fillna(0)).round(2),
                "Inkoopbedrag": (accv["aantal"].fillna(0) * accv["inkoopprijs"].fillna(0)).round(2),
                "Brutowinst": ((accv["aantal"].fillna(0) * accv["verkoopprijs"].fillna(0)) - (accv["aantal"].fillna(0) * accv["inkoopprijs"].fillna(0))).round(2),
            })
            return pd.concat([bike_fact, acc_fact], ignore_index=True)

        # Fact_Onderhoud: onderhoudsfeiten opbouwen
        def make_fact_onderhoud_df() -> pd.DataFrame:
            ond = read("Onderhoud").merge(
                read("Onderhoud_Monteur")[["monteurnr","filiaal","uurloon"]],
                left_on="monteur", right_on="monteurnr"
            )
            return pd.DataFrame({
                "OnderhoudNr": ond["onderhoudnr"],
                "ProductType": "Fiets",
                "ProductNr": ond["fiets"],
                "MonteurNr": ond["monteur"],
                "FiliaalNr": ond["filiaal"],
                "Datum": ond["datum"].astype(str).str[:10],
                "StartTijd": ond["starttijd"].apply(normalize_time),
                "EindTijd": ond["eindtijd"].apply(normalize_time),
                "AantalOnderhoud": 1,
                "OnderhoudsduurMin": ond.apply(lambda r: minutes_between(r["starttijd"], r["eindtijd"]), axis=1),
                "Arbeidskosten": ond.apply(
                    lambda r: round((minutes_between(r["starttijd"], r["eindtijd"]) / 60.0) * float(r["uurloon"]), 2)
                    if pd.notna(r["uurloon"]) else 0,
                    axis=1
                ),
            })

        # Bouw het uiteindelijke resultaat op
        result = {
            "Dim_Product": make_product_df(),
            "Dim_Partner": make_partner_df(),
            "Dim_Klant": make_klant_df(),
            "Dim_Monteur": make_monteur_df(),
            "Dim_Filiaal": make_filiaal_df(),
            "Dim_Datum": make_datum_df(),
            "Dim_Tijd": make_tijd_df(),
            "Fact_Inkoop": make_fact_inkoop_df(),
            "Fact_Verkoop": make_fact_verkoop_df(),
            "Fact_Onderhoud": make_fact_onderhoud_df(),
        }

        return result

    finally:
        # Dit garandeert dat de database niet gelocked blijft als een van de DataFrames crasht
        con.close()

### SCD‑detectie en updatefuncties

In [133]:
def detect_scd_changes(source: pd.DataFrame, target_current: pd.DataFrame, business_keys: list[str], compare_cols: list[str]) -> dict[str, pd.DataFrame]:
    # Bepaalt welke bronrijen nieuw, gewijzigd of ongewijzigd zijn
    # ten opzichte van de actuele rijen in het Data Warehouse (SCD Type 2).

    if target_current.empty:
        return {
            "new": source.copy().reset_index(drop=True),
            "changed": pd.DataFrame(columns=source.columns),
            "unchanged": pd.DataFrame(columns=source.columns),
        }

    # Voeg de twee DataFrames samen op de business keys
    merged = source.merge(target_current, on=business_keys, how="left",
                          suffixes=("_src", "_tgt"), indicator=True)

    new_mask = merged["_merge"] == "left_only"
    both_mask = merged["_merge"] == "both"

    # Bepaal verschil per kolom: leeg/NaN wordt gelijk behandeld
    if compare_cols:
        diffs = []
        for col in compare_cols:
            src_col = merged[f"{col}_src"]
            tgt_col = merged[f"{col}_tgt"]
            # Voorkomt vals-positieven wanneer beide waarden NULL/NaN zijn
            diff = ~((src_col == tgt_col) | (src_col.isna() & tgt_col.isna()))
            diffs.append(diff)

        diff_df = pd.concat(diffs, axis=1)
        changed_mask = both_mask & diff_df.any(axis=1)
    else:
        changed_mask = pd.Series(False, index=merged.index)

    unchanged_mask = both_mask & ~changed_mask

    # Geoptimaliseerde extractie zonder Python for-loop
    def extract(mask: pd.Series) -> pd.DataFrame:
        # Selecteer alleen de business keys en de _src kolommen
        cols_to_keep = business_keys + [f"{c}_src" for c in source.columns if c not in business_keys]
        extracted = merged.loc[mask, cols_to_keep].copy()

        # Hernoem de kolommen terug naar hun originele naam (verwijder '_src')
        rename_dict = {f"{c}_src": c for c in source.columns if c not in business_keys}
        return extracted.rename(columns=rename_dict).reset_index(drop=True)

    return {
        "new": extract(new_mask),
        "changed": extract(changed_mask),
        "unchanged": extract(unchanged_mask),
    }

In [134]:
def upsert_scd1(con: sqlite3.Connection, table: str, source_df: pd.DataFrame, business_keys: list[str], compare_cols: list[str]) -> dict[str, int]:
    # SCD Type 1: nieuwe rijen invoegen en gewijzigde attributen overschrijven (Bulk Operatie).

    # Een snelle check om een lege query te voorkomen
    if source_df.empty:
        return {"new": 0, "changed": 0, "unchanged": 0}

    cols_to_select = business_keys + compare_cols
    target_df = pd.read_sql_query(f"SELECT {', '.join(cols_to_select)} FROM {table}", con)

    deltas = detect_scd_changes(source_df, target_df, business_keys, compare_cols)
    cur = con.cursor()

    # 1. Vervang pandas NaN door None (NULL in de database) voor de hele dataframes in één keer
    new_df = deltas["new"].where(pd.notna(deltas["new"]), None)
    changed_df = deltas["changed"].where(pd.notna(deltas["changed"]), None)

    # 2. Bulk Insert nieuwe rijen
    if not new_df.empty:
        insert_sql = f"INSERT INTO {table} ({', '.join(source_df.columns)}) VALUES ({', '.join(['?'] * len(source_df.columns))})"
        # Converteer het DataFrame naar een lijst van tuples voor executemany
        insert_values = [tuple(x) for x in new_df[source_df.columns].to_numpy()]
        cur.executemany(insert_sql, insert_values)

    # 3. Bulk Update gewijzigde rijen
    if not changed_df.empty and compare_cols:
        set_clause = ", ".join([f"{col}=?" for col in compare_cols])
        where_clause = " AND ".join([f"{key}=?" for key in business_keys])
        update_sql = f"UPDATE {table} SET {set_clause} WHERE {where_clause}"

        # Zet de kolommen in de exacte volgorde voor de SET en WHERE clausules
        update_cols = compare_cols + business_keys
        update_values = [tuple(x) for x in changed_df[update_cols].to_numpy()]
        cur.executemany(update_sql, update_values)

    con.commit()

    return {k: len(v) for k, v in deltas.items()}

In [135]:
def upsert_scd2(con: sqlite3.Connection, table: str, source_df: pd.DataFrame, business_keys: list[str], compare_cols: list[str], load_ts: str) -> dict[str, int]:
    # SCD Type 2 (Bulk Operatie):
    # Nieuwe rijen worden ingevoegd. Bij bestaande rijen met wijzigingen
    # wordt de oude rij gesloten (ValidTo) en de nieuwe toegevoegd.

    if source_df.empty:
        return {"new": 0, "changed": 0, "unchanged": 0}

    # 1. Haal de huidige actieve rijen op
    cols_to_select = business_keys + compare_cols
    target_df = pd.read_sql_query(f"SELECT {', '.join(cols_to_select)} FROM {table} WHERE IsCurrent = 1", con)
    current_df = target_df[business_keys + compare_cols]

    # 2. Detecteer verschillen
    deltas = detect_scd_changes(source_df, current_df, business_keys, compare_cols)
    cur = con.cursor()

    # Vervang NaN door None voor schone database inserts (NULL)
    new_df = deltas["new"].where(pd.notna(deltas["new"]), None)
    changed_df = deltas["changed"].where(pd.notna(deltas["changed"]), None)

    # 3. Bulk UPDATE: Sluit de oude, gewijzigde rijen
    if not changed_df.empty:
        close_sql = (
            f"UPDATE {table} SET ValidTo = ?, IsCurrent = 0 "
            f"WHERE {' AND '.join([f'{k}=?' for k in business_keys])} AND IsCurrent = 1"
        )

        # Maak een lijst van tuples met parameters: [load_ts, business_key1, business_key2...]
        update_data = changed_df[business_keys].copy()
        update_data.insert(0, 'ValidTo', load_ts)
        update_values = [tuple(x) for x in update_data.to_numpy()]

        cur.executemany(close_sql, update_values)

    # 4. Bulk INSERT: Voeg nieuwe records én de nieuwe versies van gewijzigde records toe
    to_insert_df = pd.concat([new_df, changed_df], ignore_index=True)

    if not to_insert_df.empty:
        insert_cols = list(source_df.columns) + ["ValidFrom", "ValidTo", "IsCurrent"]
        insert_sql = f"INSERT INTO {table} ({', '.join(insert_cols)}) VALUES ({', '.join(['?'] * len(insert_cols))})"

        # Voeg de meta-kolommen in bulk toe aan het DataFrame
        to_insert_df["ValidFrom"] = load_ts
        to_insert_df["ValidTo"] = None
        to_insert_df["IsCurrent"] = 1

        # Zet om naar tuples in de exacte kolomvolgorde
        insert_values = [tuple(x) for x in to_insert_df[insert_cols].to_numpy()]
        cur.executemany(insert_sql, insert_values)

    con.commit()
    return {k: len(v) for k, v in deltas.items()}

In [136]:
def get_surrogate_mapping(con: sqlite3.Connection, table: str, natural_keys: list[str], surrogate: str, current_only: bool = False) -> dict[tuple, int]:
    # Haalt een razendsnelle mapping op van natuurlijke sleutels naar surrogaatsleutels.
    # Essentieel voor het vullen van de foreign keys in Fact-tabellen.

    sql = f"SELECT {', '.join(natural_keys)}, {surrogate} FROM {table}"
    if current_only:
        sql += " WHERE IsCurrent = 1"

    df = pd.read_sql_query(sql, con)

    if df.empty:
        return {}

    if len(natural_keys) == 1:
        keys = [(x,) for x in df[natural_keys[0]]]
    else:
        keys = list(df[natural_keys].itertuples(index=False, name=None))

    vals = df[surrogate].tolist()

    # Zip combineert de twee lijsten direct tot een dictionary
    return dict(zip(keys, vals))

In [137]:
def add_lookup(df: pd.DataFrame, new_col: str, mapping: dict[tuple, int], key_cols: list[str]) -> pd.DataFrame:
    # Voegt een surrogaatsleutel (Foreign Key) toe aan een DataFrame
    # met behulp van een sterk geoptimaliseerde dictionary lookup.

    if df.empty:
        df[new_col] = None
        return df

    # 1. Genereer razendsnel de tupels voor de lookup
    if len(key_cols) == 1:
        # Voor enkele sleutels forceren we een tuple structuur zoals (101,)
        keys = pd.Series([(x,) for x in df[key_cols[0]]])
    else:
        # Voor meerdere sleutels gebruiken we het sterk geoptimaliseerde itertuples
        keys = pd.Series(list(df[key_cols].itertuples(index=False, name=None)))

    # 2. Gebruik pandas' ingebouwde .map() voor een directe hash-table lookup
    df[new_col] = keys.map(mapping)

    return df

In [138]:
def insert_new_facts(con: sqlite3.Connection, table: str, source_df: pd.DataFrame,
                     business_keys: list[str]) -> dict[str, int]:
    # Voegt alleen nieuwe fact-rijen toe aan de database (geen updates).
    # Bestaande rijen worden overgeslagen op basis van de opgegeven business keys.

    if source_df.empty:
        return {"new": 0, "existing": 0}

    # 1. Controleer of de tabel bestaat (Leesbaarheid verbeterd)
    table_exists = con.execute(
        f"SELECT count(name) FROM sqlite_master WHERE type='table' AND name='{table}'"
    ).fetchone()[0]

    if table_exists:
        existing = pd.read_sql_query(f"SELECT {', '.join(business_keys)} FROM {table}", con)
    else:
        existing = pd.DataFrame(columns=business_keys)

    # 2. Bepaal welke rijen echt nieuw zijn via een left join
    merged = source_df.merge(existing, on=business_keys, how="left", indicator=True)
    new_rows = merged[merged["_merge"] == "left_only"][source_df.columns].copy()

    # 3. Bulk insert de nieuwe rijen
    if not new_rows.empty:
        cur = con.cursor()

        # Vervang pandas NaN door None (wordt NULL in database) in één razendsnelle actie
        new_rows_clean = new_rows.where(pd.notna(new_rows), None)

        insert_sql = f"INSERT INTO {table} ({', '.join(source_df.columns)}) VALUES ({', '.join(['?'] * len(source_df.columns))})"

        # Gebruik de veel snellere executemany() methode met numpy tuples
        insert_values = [tuple(x) for x in new_rows_clean.to_numpy()]
        cur.executemany(insert_sql, insert_values)
        con.commit()

    return {"new": len(new_rows), "existing": len(source_df) - len(new_rows)}

### ETL uitvoeren (dimension load, fact load en overzicht)

In [139]:
def run_etl(sdm_path: Path, dws_path: Path) -> pd.DataFrame:

    # Orkestreert het volledige ETL-proces:
    # 1. Schema voorbereiden
    # 2. Data extraheren
    # 3. Dimensies laden (SCD1 & SCD2)
    # 4. Mappings ophalen
    # 5. Feitentabellen laden met surrogaatsleutels

    # Stap 1: maak schema aan
    prepare_target_schema(dws_path)

    # Stap 2: extract
    src = extract_sdm_data(sdm_path)
    con = connect_db(dws_path)

    try:
        results = {}
        load_ts = dt.datetime.now().replace(microsecond=0).isoformat()

        # Configureer per dimensie SCD‑type en sleutelkolommen
        dim_cfg = {
            "Dim_Datum":  (1, ["Datum"], ["Dag", "Maand", "Jaar", "Kwartaal", "Weekdag", "Seizoen", "IsWeekend"]),
            "Dim_Tijd":   (1, ["Tijd"], ["Uur", "Minuut", "Dagdeel"]),
            "Dim_Partner":(1, ["PartnerType", "PartnerNr"], ["Naam", "Adres", "Plaats"]),
            "Dim_Filiaal":(1, ["FiliaalNr"], ["Naam", "Adres", "Provincie"]),
            "Dim_Monteur":(1, ["MonteurNr"], ["Naam", "Woonplaats", "Uurloon"]),
            "Dim_Product":(2, ["ProductType", "ProductNr"], ["Naam", "Merk", "Soort", "Type", "Kleur", "Standaardprijs", "Inkoopprijs"]),
            "Dim_Klant":  (2, ["KlantNr"], ["Naam", "Adres", "Woonplaats", "Geslacht", "Geboortedatum", "Leeftijd", "Leeftijdscategorie"]),
        }

        # Stap 3: laad dimensies
        for table, (scd_type, keys, attrs) in dim_cfg.items():
            df = src[table].copy()
            if scd_type == 1:
                res = upsert_scd1(con, table, df, keys, attrs)
            else:
                res = upsert_scd2(con, table, df, keys, attrs, load_ts)
            results[table] = res

        # Stap 4: maak mappings
        maps = {
            "Product": get_surrogate_mapping(con, "Dim_Product", ["ProductType", "ProductNr"], "ProductKey", current_only=True),
            "Partner": get_surrogate_mapping(con, "Dim_Partner", ["PartnerType", "PartnerNr"], "PartnerKey"),
            "Klant":   get_surrogate_mapping(con, "Dim_Klant", ["KlantNr"], "KlantKey", current_only=True),
            "Monteur": get_surrogate_mapping(con, "Dim_Monteur", ["MonteurNr"], "MonteurKey"),
            "Filiaal": get_surrogate_mapping(con, "Dim_Filiaal", ["FiliaalNr"], "FiliaalKey"),
            "Datum":   get_surrogate_mapping(con, "Dim_Datum", ["Datum"], "DatumKey"),
            "Tijd":    get_surrogate_mapping(con, "Dim_Tijd", ["Tijd"], "TijdKey"),
        }

        # Stap 5: laad fact_tabellen

        # --- Fact_Inkoop ---
        fact_inkoop = src["Fact_Inkoop"].copy()
        add_lookup(fact_inkoop, "ProductKey", maps["Product"], ["ProductType", "ProductNr"])
        add_lookup(fact_inkoop, "PartnerKey", maps["Partner"], ["PartnerType", "PartnerNr"])
        add_lookup(fact_inkoop, "DatumKey", maps["Datum"], ["Datum"]) # Lambda vervangen!

        fact_inkoop = fact_inkoop[["InkoopNr", "ProductKey", "PartnerKey", "DatumKey", "Aantal", "Inkoopprijs", "Inkoopbedrag", "KortingBedrag"]]
        results["Fact_Inkoop"] = insert_new_facts(con, "Fact_Inkoop", fact_inkoop, ["InkoopNr"])

        # --- Fact_Verkoop ---
        fact_verkoop = src["Fact_Verkoop"].copy()
        add_lookup(fact_verkoop, "ProductKey", maps["Product"], ["ProductType", "ProductNr"])
        add_lookup(fact_verkoop, "KlantKey", maps["Klant"], ["KlantNr"])
        add_lookup(fact_verkoop, "MonteurKey", maps["Monteur"], ["MonteurNr"])
        add_lookup(fact_verkoop, "FiliaalKey", maps["Filiaal"], ["FiliaalNr"])
        add_lookup(fact_verkoop, "DatumKey", maps["Datum"], ["Datum"]) # Lambda vervangen!

        fact_verkoop = fact_verkoop[["VerkoopNr", "ProductKey", "KlantKey", "MonteurKey", "FiliaalKey", "DatumKey", "Aantal", "Verkoopprijs", "Omzet", "Inkoopbedrag", "Brutowinst"]]
        results["Fact_Verkoop"] = insert_new_facts(con, "Fact_Verkoop", fact_verkoop, ["VerkoopNr"])

        # --- Fact_Onderhoud ---
        fact_onderhoud = src["Fact_Onderhoud"].copy()
        add_lookup(fact_onderhoud, "ProductKey", maps["Product"], ["ProductType", "ProductNr"])
        add_lookup(fact_onderhoud, "MonteurKey", maps["Monteur"], ["MonteurNr"])
        add_lookup(fact_onderhoud, "FiliaalKey", maps["Filiaal"], ["FiliaalNr"])
        add_lookup(fact_onderhoud, "DatumKey", maps["Datum"], ["Datum"])           # Lambda vervangen!
        add_lookup(fact_onderhoud, "StartTijdKey", maps["Tijd"], ["StartTijd"])    # Lambda vervangen!
        add_lookup(fact_onderhoud, "EindTijdKey", maps["Tijd"], ["EindTijd"])      # Lambda vervangen!

        fact_onderhoud = fact_onderhoud[["OnderhoudNr", "ProductKey", "MonteurKey", "FiliaalKey", "DatumKey", "StartTijdKey", "EindTijdKey", "AantalOnderhoud", "OnderhoudsduurMin", "Arbeidskosten"]]
        results["Fact_Onderhoud"] = insert_new_facts(con, "Fact_Onderhoud", fact_onderhoud, ["OnderhoudNr"])

        # Resultaat samenvatten als DataFrame met behulp van list comprehension
        rows = [
            {"Tabel": table, "Soort": op, "Aantal": count}
            for table, res in results.items()
            for op, count in res.items()
        ]
        return pd.DataFrame(rows)

    finally:
        # Garandeert dat de database verbinding ALTIJD sluit
        con.close()

In [140]:
# Zorg ervoor dat de map 'database' daadwerkelijk bestaat om SQLite fouten te voorkomen
base_dir = Path.cwd()
sdm_path = base_dir / "database" / "SDM.db"
dws_path = base_dir / "database" / "DWS.db"
dws_path.parent.mkdir(parents=True, exist_ok=True)

print("ETL-proces wordt gestart... Dit kan even duren afhankelijk van de datahoeveelheid.")

try:
    # Voer de ETL uit en toon een overzicht
    overzicht = run_etl(sdm_path, dws_path)
    overview_sorted = overzicht.sort_values(["Tabel", "Soort"]).reset_index(drop=True)

    print("ETL-proces succesvol afgerond! Hier is het overzicht:")
    display(overview_sorted)

except Exception as e:
    print(f"CRITISCHE FOUT tijdens ETL-proces: {e}")

ETL-proces wordt gestart... Dit kan even duren afhankelijk van de datahoeveelheid.
ETL-proces succesvol afgerond! Hier is het overzicht:


,Tabel,Soort,Aantal
0,Dim_Datum,changed,0
1,Dim_Datum,new,1
2,Dim_Datum,unchanged,201
3,Dim_Filiaal,changed,1
4,Dim_Filiaal,new,0
5,Dim_Filiaal,unchanged,4
6,Dim_Klant,changed,0
7,Dim_Klant,new,0
8,Dim_Klant,unchanged,25
9,Dim_Monteur,changed,0


# TEST

In [141]:
# con = sqlite3.connect(Path.cwd() / "database" / "SDM.db")
#
# try:
#     con.executescript("""
#         UPDATE Fietsverkoop_Filiaal SET naam = 'Nieuwe Naam ' || RANDOM() WHERE filiaalnr = (SELECT MIN(filiaalnr) FROM Fietsverkoop_Filiaal);
#
#         UPDATE Fietsverkoop_Klant SET woonplaats = 'Nieuw-Dorp' WHERE klantnr = (SELECT MIN(klantnr) FROM Fietsverkoop_Klant);
#
#         INSERT INTO Fietsverkoop_Fiets_Verkoop (fiets_verkoopnr, fiets, klant, monteur, datum, aantal, verkoopprijs)
#         SELECT COALESCE(MAX(fiets_verkoopnr), 0)+1, 1, 1, 1, '2026-03-31', 1, 999.99 FROM Fietsverkoop_Fiets_Verkoop;
#     """)
#     print("✅ Testdata toegevoegd! Draai je ETL-cel (run_etl) nu nog een keer.")
# finally:
#     con.close()

In [ ]:
# # Configuratie van paden
# base_dir = Path.cwd()
# sdm_path = base_dir / "database" / "SDM.db"
# dws_path = base_dir / "database" / "DWS.db"
#
# # Validatie van de doel-directory
# dws_path.parent.mkdir(parents=True, exist_ok=True)
#
# # --- Demonstratie Header ---
# print("-" * 70)
# print("UITVOERING DATA WAREHOUSE ETL-PIJPLIJN")
# print("-" * 70)
# print(f"Bron database (SDM) : {sdm_path.name}")
# print(f"Doel database (DWH) : {dws_path.name}")
# print("-" * 70)
#
# print("\nInitialiseren van het ETL-proces...")
# print("Data extraheren, schema's valideren en transformaties toepassen...")
#
# # Start de prestatiemeting
# start_time = time.time()
#
# try:
#     # Uitvoeren van het kernproces
#     overzicht = run_etl(sdm_path, dws_path)
#
#     # Stop de prestatiemeting
#     eind_tijd = time.time()
#     doorlooptijd = eind_tijd - start_time
#
#     # Sorteren van de resultaten voor een overzichtelijke weergave
#     overview_sorted = overzicht.sort_values(["Tabel", "Soort"]).reset_index(drop=True)
#
#     # --- Resultaten en Evaluatie ---
#     print(f"\nETL-proces succesvol afgerond in {doorlooptijd:.4f} seconden.")
#     print("\n" + "=" * 70)
#     print("RESULTATEN OVERZICHT (Mutatiedetectie & Slowly Changing Dimensions)")
#     print("=" * 70)
#
#     # Contextuele uitleg voor de demonstratie
#     print("Toelichting bij de mutatiesoorten:")
#     print("  - 'new'       : Volledig nieuwe records, succesvol ingeladen in het DWH.")
#     print("  - 'changed'   : Gewijzigde brondata. Deze records zijn overschreven (SCD Type 1)")
#     print("                  of historisch afgesloten met een nieuwe actieve regel (SCD Type 2).")
#     print("  - 'existing' /")
#     print("    'unchanged' : Ongewijzigde data. Deze records zijn veilig genegeerd.")
#     print("                  (Dit toont de idempotente en deduplicerende werking van de code aan).\n")
#
#     # Weergave van de data-tabel
#     display(overview_sorted)
#
# except Exception as e:
#     print("\nKRITISCHE FOUT: Het ETL-proces is onverwacht afgebroken.")
#     print(f"Technische details: {e}")
#     print("-" * 70)